# Purpose :

Take data from /data, and preprocess it in /data_prepared.

In [100]:
# Imports and data loading
import pandas as pd
import os
from sklearn.preprocessing import MinMaxScaler

# Loading all tables
abbrev = pd.read_csv('./data/abbrev.csv')
awardsCoaches = pd.read_csv('./data/AwardsCoaches.csv')
awardsMisc = pd.read_csv('./data/AwardsMisc.csv')
awardsPlayers = pd.read_csv('./data/AwardsPlayers.csv')
coaches = pd.read_csv('./data/Coaches.csv')
combinedShutouts = pd.read_csv('./data/CombinedShutouts.csv')
goalies = pd.read_csv('./data/Goalies.csv')
goaliesSC = pd.read_csv('./data/GoaliesSC.csv')
goaliesShootout = pd.read_csv('./data/GoaliesShootout.csv')
hof = pd.read_csv('./data/HOF.csv')
master = pd.read_csv('./data/Master.csv')
scoring = pd.read_csv('./data/Scoring.csv')
scoringSC = pd.read_csv('./data/ScoringSC.csv')
scoringShootout = pd.read_csv('./data/ScoringShootout.csv')
scoringSup = pd.read_csv('./data/ScoringSup.csv')
seriesPost = pd.read_csv('./data/SeriesPost.csv')
teams = pd.read_csv('./data/Teams.csv')
teamsHalf = pd.read_csv('./data/TeamsHalf.csv')
teamSplits = pd.read_csv('./data/TeamSplits.csv')
teamsPost = pd.read_csv('./data/TeamsPost.csv')
teamsSC = pd.read_csv('./data/TeamsSC.csv')
teamsVsTeam = pd.read_csv('./data/TeamVsTeam.csv')

# Select Data

We only keep years above 1979 (year in which there is only NHL league) and NHL data.

In [101]:
YEAR = 1979

awardsCoaches       = awardsCoaches[awardsCoaches['year'] > YEAR]
awardsMisc          = awardsMisc[awardsMisc['year'] > YEAR]
awardsPlayers       = awardsPlayers[awardsPlayers['year'] > YEAR]
coaches             = coaches[(coaches['year'] > YEAR) & (coaches['lgID'] == 'NHL')]
combinedShutouts    = combinedShutouts[combinedShutouts['year'] > YEAR]
goalies             = goalies[goalies['year'] > YEAR]
goaliesSC           = goaliesSC[goaliesSC['year'] > YEAR]
goaliesShootout     = goaliesShootout[goaliesShootout['year'] > YEAR]
hof                 = hof[hof['year'] > YEAR]
scoring             = scoring[scoring['year'] > YEAR]
scoringSC           = scoringSC[scoringSC['year'] > YEAR]
scoringShootout     = scoringShootout[scoringShootout['year'] > YEAR]
scoringSup          = scoringSup[scoringSup['year'] > YEAR]
seriesPost          = seriesPost[seriesPost['year'] > YEAR]
teams               = teams[(teams['year'] > YEAR) & (teams['lgID'] == 'NHL')]
teamsHalf           = teamsHalf[teamsHalf['year'] > YEAR]
teamSplits          = teamSplits[teamSplits['year'] > YEAR]
teamsPost           = teamsPost[teamsPost['year'] > YEAR]
teamsSC             = teamsSC[teamsSC['year'] > YEAR]
teamsVsTeam         = teamsVsTeam[teamsVsTeam['year'] > YEAR]

# Clean Data

Remove the tables that are not needed for this analysis.

In [102]:
awardsMisc  = None
goaliesSC   = None
scoringSC   = None
scoringSup  = None
teamsHalf   = None
teamsSC     = None

# Construct Data

Add the target for an in-season coach change (1 if any coach change occurs during the season, 0 otherwise).

In [103]:
teams = teams.copy()
coaches = coaches.copy()

# Associate each coach record with a franchise.
team_franchises = teams[["tmID", "year", "franchID"]].drop_duplicates()
coaches_f = coaches.merge(team_franchises, on=["tmID", "year"], how="left")

# A change occurred if a coach has a stint > 1 during the season
changes = coaches_f.groupby(["franchID", "year"])["stint"].max().reset_index()
changes["coach_changed"] = (changes["stint"] > 1).astype(int)
changes = changes[["franchID", "year", "coach_changed"]]

teams = teams.merge(changes, on=["franchID", "year"], how="left")
teams["coach_changed"] = teams["coach_changed"].fillna(0).astype(int)

print("In-season coach changes:")
print(int(teams["coach_changed"].sum()))
print("\nNo in-season coach change:")
print(int((teams["coach_changed"] == 0).sum()))

In-season coach changes:
140

No in-season coach change:
592


Add a column for teams that shows the global ranking (1-15) in the current conference (Eastern or Western depending on the team).

In [104]:
teams = teams.sort_values(['year', 'confID', 'Pts', 'tmID'],
                          ascending=[True, True, False, True]).copy()
teams['conference_rank'] = (teams.groupby(['year', 'confID']).cumcount() + 1).astype(int)
teams['division_rank'] = (teams.groupby(['year', 'divID']).cumcount() + 1).astype(int)

teams = teams.sort_values(['tmID', 'year'])
teams['prev_conference_rank'] = teams.groupby('tmID')['conference_rank'].shift(1)
teams['prev_division_rank'] = teams.groupby('tmID')['division_rank'].shift(1)

YEAR_TO_DISPLAY = 2008
print(f"Conference ranking for {YEAR_TO_DISPLAY}:")
print(
    teams[teams['year'] == YEAR_TO_DISPLAY]
    [['prev_conference_rank', 'conference_rank', 'confID', 'tmID', 'name', 'Pts']]
    .sort_values(['confID', 'conference_rank'])
    .to_string(index=False)
)

Conference ranking for 2008:
 prev_conference_rank  conference_rank confID tmID                  name  Pts
                  6.0                1     EC  BOS         Boston Bruins  116
                  8.0                2     EC  WAS   Washington Capitals  108
                  3.0                3     EC  NJD     New Jersey Devils  106
                  5.0                4     EC  PHI   Philadelphia Flyers   99
                  2.0                5     EC  PIT   Pittsburgh Penguins   99
                  9.0                6     EC  CAR   Carolina Hurricanes   97
                  4.0                7     EC  NYR      New York Rangers   95
                 11.0                8     EC  FLO      Florida Panthers   93
                  1.0                9     EC  MTL    Montreal Canadiens   93
                 10.0               10     EC  BUF        Buffalo Sabres   91
                  7.0               11     EC  OTT       Ottawa Senators   83
                 12.0              

## Features for coach-change prediction

These variables use only previous seasons, so the current season target is never used as an input. We keep only the most relevant features to avoid overfitting.

In [105]:
teams = teams.sort_values(['franchID', 'year']).copy()
g = teams.groupby('franchID', sort=False)

# A. Previous season context
teams['prev_Pts'] = g['Pts'].shift(1)
teams['prev_rank'] = g['rank'].shift(1)
teams['prev_playoff'] = g['playoff'].shift(1).fillna('').ne('').astype(int)

# B. Trends (Variations over 1 year)
teams['pts_change_1y'] = teams['prev_Pts'] - g['Pts'].shift(2)
teams['rank_change_1y'] = teams['prev_rank'] - g['rank'].shift(2)
teams['goal_diff_1y'] = (g['GF'].shift(1) - g['GA'].shift(1)) - (g['GF'].shift(2) - g['GA'].shift(2))

# C. Coach History (Memory)
teams['prev_coach_changed'] = g['coach_changed'].shift(1).fillna(0)
teams['changes_last_3y'] = g['coach_changed'].transform(
    lambda s: s.shift(1).rolling(3, min_periods=1).sum()
).fillna(0)

# Coach tenure (approximated by consecutive seasons)
teams['coach_tenure'] = teams.groupby(['franchID', 'coach_changed']).cumcount()

teams = teams.sort_values(['year', 'tmID']).reset_index(drop=True)

# Format Data

Normalize the data and save the specific dataset for coach predictions.

In [106]:
# List of features to keep for the final dataset
features = [
    'prev_Pts', 'prev_rank', 'prev_playoff', 
    'pts_change_1y', 'rank_change_1y', 'goal_diff_1y',
    'prev_coach_changed', 'changes_last_3y', 'coach_tenure'
]

# Fill NaNs (first years of each franchise) with the median
teams[features] = teams[features].fillna(teams[features].median())

# Normalization
scaler = MinMaxScaler()
teams[features] = scaler.fit_transform(teams[features])

# Save the cleaned dataset
cols_to_keep = ['franchID', 'year', 'coach_changed'] + features
teams_clean = teams[cols_to_keep].copy()

os.makedirs('./data_prepared', exist_ok=True)
teams_clean.to_csv('./data_prepared/teams_final.csv', index=False)
print(f"File saved: ./data_prepared/teams_final.csv")
print(f"Number of features: {len(features)}")

File saved: ./data_prepared/teams_final.csv
Number of features: 9
